In [2]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime, timedelta
from itertools import product
from RS.PreRun import PreRun, PostRun
from sklearn.linear_model import LinearRegression


# path to data
read_path = "../../../data_ds_project/parquet_cleaned_energy"
systems_cleaned = pd.read_csv("../../data/core/systems_cleaned.csv")
relevant_system_pairs = [(10, None), (50, None), (51, None)]

In [3]:
naive_folder_str = './RS/naive_errors/'
naive_file_suffix = '_naive_errors.csv'
lin_folder_str = './RS/linreg_errors/'
lin_file_suffix = '_linreg_errors.csv'
lin_fourier_suffix = '_linreg_errors_fourier.csv'
prophet_folder_str = './RS/prophet_errors/'
prophet_file_suffix = '_prophet_errors.csv'
sarimax_folder_str = './RS/sarimax_errors/'
sarimax_file_suffix = '_sarimax_errors.csv'
xgboost_folder_str = './CEB/xgboost_results/'
xgboost_folder_suffix = '.csv'
lightgbm_folder_str = './CEB/lightgbm_results/'
lightgbm_folder_suffix = '.csv'

model_tuple = ('naive', 'lin_reg', 'lin_reg_with_fourier', 'prophet', 'sarimax', 'xgboost', 'lightgbm')
model_folder = (naive_folder_str, lin_folder_str, lin_folder_str, prophet_folder_str,
                sarimax_folder_str, xgboost_folder_str, lightgbm_folder_str)
model_suffix = (naive_file_suffix, lin_file_suffix, lin_fourier_suffix, prophet_file_suffix,
                sarimax_file_suffix, xgboost_folder_suffix, lightgbm_folder_suffix)
sarimax_col_names = [
    '2,0,0', '2,0,1', '3,0,0', '3,0,1'
]
xgboost_col_names = [
    '(31, 5, 0.1, 1.0, 0.8)', '(31, 7, 0.1, 0.8, 0.8)', '(31, 10, 0.1, 0.8, 0.8)'
]

lightgbm_col_names = [
    '(31, -1, 0.1, 100, 0.8, 0.8)',
    '(31, -1, 0.1, 100, 1.0, 0.8)'
]


## First comparison -- whole training set

In [4]:
def read_results_summaries(system_id: int):
    my_cols = ['per_model_mean', 'per_model_std', 'per_model_min', 'per_model_25p', 'per_model_median', 'per_model_75p', 'per_model_max']
    per_model_results = []
    for j in range(7):
        model_name = model_tuple[j]
        model_results = pd.read_csv(f'{model_folder[j]}{system_id}_None{model_suffix[j]}')
        if model_name == 'xgboost':
            model_results = model_results[xgboost_col_names]
        elif model_name == 'lightgbm':
            model_results = model_results[lightgbm_col_names]
        elif model_name == 'sarimax':
            model_results = model_results[sarimax_col_names]
        model_results = model_results.rename(columns={
            col_name: f'{model_name}_{col_name}' for col_name in model_results.columns
        })
        model_results = model_results.transpose()
        ordinary_cols = model_results.columns
        model_results.loc[:, 'per_model_mean'] = model_results[ordinary_cols].mean(axis=1)
        model_results.loc[:, 'per_model_std'] = model_results[ordinary_cols].std(axis=1)
        model_results.loc[:, 'per_model_min'] = model_results[ordinary_cols].min(axis=1)
        model_results.loc[:, 'per_model_25p'] = model_results[ordinary_cols].quantile(q=0.25, axis=1)
        model_results.loc[:, 'per_model_median'] = model_results[ordinary_cols].quantile(q=0.5, axis=1)
        model_results.loc[:, 'per_model_75p'] = model_results[ordinary_cols].quantile(q=0.75, axis=1)
        model_results.loc[:, 'per_model_max'] = model_results[ordinary_cols].max(axis=1)
        per_model_results.append(model_results[my_cols])
    total_results = pd.concat(per_model_results)
    return total_results


### System 10

In [4]:
read_results_summaries(10)

,per_model_mean,per_model_std,per_model_min,per_model_25p,per_model_median,per_model_75p,per_model_max
naive_error,0.044598,0.057223,0.000106,0.008121,0.020224,0.055983,0.323554
lin_reg_error,0.033743,0.043212,0.000505,0.006897,0.018895,0.044661,0.370300
lin_reg_with_fourier_error,0.033711,0.043174,0.000347,0.006709,0.018773,0.045067,0.369017
"prophet_(0.1, 10, 20, 2)",0.029864,0.034978,0.001236,0.008429,0.015200,0.037130,0.182796
"prophet_(0.5, 10, 20, 2)",0.030024,0.035139,0.001230,0.008471,0.015158,0.038870,0.181705
"sarimax_2,0,0",0.025480,0.032481,0.000259,0.005789,0.012818,0.031939,0.133287
"sarimax_2,0,1",0.025889,0.033425,0.000240,0.005681,0.013763,0.032576,0.135753
"sarimax_3,0,0",0.025594,0.032811,0.000240,0.006000,0.012696,0.031869,0.135702
"sarimax_3,0,1",0.025968,0.032693,0.000240,0.005986,0.014436,0.031832,0.135757
"xgboost_(31, 5, 0.1, 1.0, 0.8)",0.038852,0.038669,0.000292,0.013059,0.025964,0.052271,0.262199


Sarimax best, prophet second-best

### System 50

In [5]:
read_results_summaries(50)

,per_model_mean,per_model_std,per_model_min,per_model_25p,per_model_median,per_model_75p,per_model_max
naive_error,1.747726,2.641483,2.944007e-03,0.249494,0.630318,1.822918,16.621731
lin_reg_error,1.198927,1.607186,1.932127e-02,0.265745,0.643737,1.444014,14.255237
lin_reg_with_fourier_error,1.190300,1.608457,1.598518e-02,0.252537,0.636493,1.447720,14.219866
"prophet_(0.1, 10, 20, 2)",1.253558,1.567298,1.623266e-02,0.303753,0.633534,1.334442,7.453973
"sarimax_2,0,0",0.421169,0.574540,9.573573e-12,0.110385,0.244722,0.502947,4.652707
"sarimax_2,0,1",0.430401,0.580942,9.573573e-12,0.110290,0.252629,0.521132,4.634019
"sarimax_3,0,0",0.416980,0.571410,9.573573e-12,0.108380,0.227699,0.495708,4.626921
"sarimax_3,0,1",0.413413,0.572550,9.573573e-12,0.107507,0.222265,0.496839,4.636144
"xgboost_(31, 5, 0.1, 1.0, 0.8)",1.053864,1.212296,3.277221e-03,0.295317,0.657174,1.363442,8.290764
"xgboost_(31, 7, 0.1, 0.8, 0.8)",1.041629,1.206276,2.204182e-03,0.301800,0.648082,1.435311,8.887016


Sarimax best, XGBoost 2nd-best

### System 51

In [6]:
read_results_summaries(51)

,per_model_mean,per_model_std,per_model_min,per_model_25p,per_model_median,per_model_75p,per_model_max
naive_error,1.483751,2.052073,0.001444,0.240178,0.614046,1.778395,15.571347
lin_reg_error,1.102705,1.460331,0.029086,0.244206,0.590222,1.413839,12.430139
lin_reg_with_fourier_error,1.098716,1.460753,0.028359,0.225830,0.593590,1.430833,12.529982
"prophet_(0.1, 10, 20, 2)",0.960666,1.030279,0.061602,0.285210,0.545052,1.299444,5.326777
"sarimax_2,0,0",0.580453,0.808239,0.011611,0.148492,0.286351,0.643621,4.842274
"sarimax_2,0,1",0.586206,0.802457,0.022000,0.149586,0.290685,0.594057,4.815697
"sarimax_3,0,0",0.570796,0.806616,0.011896,0.142795,0.279711,0.607576,4.805762
"sarimax_3,0,1",0.569663,0.808612,0.011885,0.141899,0.280137,0.599693,4.817465
"xgboost_(31, 5, 0.1, 1.0, 0.8)",1.086789,1.011577,0.012522,0.426061,0.780459,1.374913,6.236039
"xgboost_(31, 7, 0.1, 0.8, 0.8)",1.100136,1.035525,0.019361,0.438710,0.756980,1.378442,6.097769


OK, Sarimax appears to be the best model-set in training!

## 2nd Comparison: 2nd half training data only (on the assumption that later, more-time-data datasets will perform better)

In [5]:
def read_results_second_half(system_id: int):
    my_cols = ['per_model_mean', 'per_model_std', 'per_model_min', 'per_model_25p', 'per_model_median', 'per_model_75p', 'per_model_max']
    per_model_results = []
    for j in range(7):
        model_name = model_tuple[j]
        model_results = pd.read_csv(f'{model_folder[j]}{system_id}_None{model_suffix[j]}')
        if model_name == 'xgboost':
            model_results = model_results[xgboost_col_names]
        elif model_name == 'lightgbm':
            model_results = model_results[lightgbm_col_names]
        elif model_name == 'sarimax':
            model_results = model_results[sarimax_col_names]
        model_results = model_results.rename(columns={
            col_name: f'{model_name}_{col_name}' for col_name in model_results.columns
        })
        # 2nd half of data
        num_entries = model_results.shape[0]
        model_results = model_results.iloc[int(num_entries/2):]
        model_results = model_results.transpose()
        ordinary_cols = model_results.columns
        model_results.loc[:, 'per_model_mean'] = model_results[ordinary_cols].mean(axis=1)
        model_results.loc[:, 'per_model_std'] = model_results[ordinary_cols].std(axis=1)
        model_results.loc[:, 'per_model_min'] = model_results[ordinary_cols].min(axis=1)
        model_results.loc[:, 'per_model_25p'] = model_results[ordinary_cols].quantile(q=0.25, axis=1)
        model_results.loc[:, 'per_model_median'] = model_results[ordinary_cols].quantile(q=0.5, axis=1)
        model_results.loc[:, 'per_model_75p'] = model_results[ordinary_cols].quantile(q=0.75, axis=1)
        model_results.loc[:, 'per_model_max'] = model_results[ordinary_cols].max(axis=1)
        per_model_results.append(model_results[my_cols])
    total_results = pd.concat(per_model_results)
    return total_results

### System 10

In [5]:
read_results_second_half(10)

,per_model_mean,per_model_std,per_model_min,per_model_25p,per_model_median,per_model_75p,per_model_max
naive_error,0.028386,0.038575,0.000346,0.005577,0.013404,0.033578,0.294026
lin_reg_error,0.028605,0.033698,0.000505,0.006269,0.015959,0.041282,0.222296
lin_reg_with_fourier_error,0.028586,0.033909,0.000347,0.006151,0.015742,0.041629,0.217507
"prophet_(0.1, 10, 20, 2)",0.028918,0.033670,0.001236,0.006046,0.013890,0.040619,0.141223
"prophet_(0.5, 10, 20, 2)",0.028946,0.033715,0.001230,0.005970,0.013843,0.040648,0.141110
"sarimax_2,0,0",0.023089,0.038674,0.000259,0.003605,0.006630,0.024616,0.133287
"sarimax_2,0,1",0.023517,0.039368,0.000240,0.003687,0.005846,0.025337,0.135753
"sarimax_3,0,0",0.023355,0.039340,0.000240,0.003813,0.006813,0.024643,0.135702
"sarimax_3,0,1",0.024142,0.039214,0.000240,0.003798,0.006758,0.027258,0.135757
"xgboost_(31, 5, 0.1, 1.0, 0.8)",0.035576,0.035977,0.000292,0.011142,0.023888,0.048493,0.184408


The baseline model and linear regression are almost as good as SARIMAX; gradient boosting has no such consistent effect.

### System 50

In [6]:
read_results_second_half(50)

,per_model_mean,per_model_std,per_model_min,per_model_25p,per_model_median,per_model_75p,per_model_max
naive_error,0.922217,1.313093,0.011960,0.189236,0.424500,1.033867,9.759413
lin_reg_error,0.899575,1.085668,0.019321,0.258666,0.550870,1.180666,11.543715
lin_reg_with_fourier_error,0.884930,1.071907,0.015985,0.241631,0.535372,1.195852,11.612355
"prophet_(0.1, 10, 20, 2)",0.791058,0.880414,0.016233,0.271869,0.436917,0.877845,4.788489
"sarimax_2,0,0",0.434003,0.495671,0.054671,0.133307,0.271171,0.511851,2.613551
"sarimax_2,0,1",0.445341,0.499589,0.053445,0.139490,0.267802,0.532956,2.587537
"sarimax_3,0,0",0.436768,0.492581,0.055146,0.144592,0.270568,0.514668,2.635536
"sarimax_3,0,1",0.429145,0.494270,0.055123,0.129675,0.257140,0.530383,2.657943
"xgboost_(31, 5, 0.1, 1.0, 0.8)",1.116589,1.028425,0.051468,0.412511,0.755211,1.563812,5.065310
"xgboost_(31, 7, 0.1, 0.8, 0.8)",1.104654,1.052008,0.039536,0.409833,0.730605,1.468399,5.447737


Still the same rankings as the full set of data.

### System 51

In [7]:
read_results_second_half(51)

,per_model_mean,per_model_std,per_model_min,per_model_25p,per_model_median,per_model_75p,per_model_max
naive_error,1.108734,1.549795,0.018573,0.199691,0.480194,1.351895,12.157881
lin_reg_error,1.053041,1.434794,0.032915,0.234769,0.565328,1.329871,12.430139
lin_reg_with_fourier_error,1.050867,1.435623,0.028359,0.216825,0.573282,1.369444,12.365689
"prophet_(0.1, 10, 20, 2)",0.901536,1.011971,0.069066,0.246216,0.524496,1.087614,4.882891
"sarimax_2,0,0",0.527761,0.587800,0.011611,0.184390,0.313270,0.658956,2.558104
"sarimax_2,0,1",0.533575,0.584183,0.022000,0.172574,0.325686,0.617846,2.543440
"sarimax_3,0,0",0.504668,0.577318,0.011896,0.181626,0.307904,0.612946,2.546899
"sarimax_3,0,1",0.505258,0.583222,0.011885,0.181375,0.306963,0.612657,2.546969
"xgboost_(31, 5, 0.1, 1.0, 0.8)",1.102413,0.966874,0.073289,0.515660,0.795826,1.377943,4.937423
"xgboost_(31, 7, 0.1, 0.8, 0.8)",1.131353,1.037971,0.031312,0.512291,0.772730,1.322765,5.548167


Ditto

Quick printout

In [8]:
my_data = pd.merge(
    left=read_results_summaries(10)[['per_model_mean', 'per_model_std']],
    right=read_results_second_half(10)[['per_model_mean', 'per_model_std']],
    left_index=True,
    right_index=True,
    how='inner',
    suffixes=('_total', '_2nd_half')
)
my_data

,per_model_mean_total,per_model_std_total,per_model_mean_2nd_half,per_model_std_2nd_half
naive_error,0.044598,0.057223,0.028386,0.038575
lin_reg_error,0.033743,0.043212,0.028605,0.033698
lin_reg_with_fourier_error,0.033711,0.043174,0.028586,0.033909
"prophet_(0.1, 10, 20, 2)",0.029864,0.034978,0.028918,0.033670
"prophet_(0.5, 10, 20, 2)",0.030024,0.035139,0.028946,0.033715
"sarimax_2,0,0",0.025480,0.032481,0.023089,0.038674
"sarimax_2,0,1",0.025889,0.033425,0.023517,0.039368
"sarimax_3,0,0",0.025594,0.032811,0.023355,0.039340
"sarimax_3,0,1",0.025968,0.032693,0.024142,0.039214
"xgboost_(31, 5, 0.1, 1.0, 0.8)",0.038852,0.038669,0.035576,0.035977
